In [1]:
!pip install -q segmentation-models-pytorch albumentations



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ====================================================
# 0. IMPORT & REPRODUCIBILITY
# ====================================================
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import torch.nn.functional as F
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ====================================================
# 1. CONFIG & PATHS
# ====================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2

BASE_DIR = "data-science-ara-7-0/dataset/dataset/"
TRAIN_IMG = os.path.join(BASE_DIR, "train/images/")
TRAIN_MASK = os.path.join(BASE_DIR, "train/mask/")
TEST_IMG = os.path.join(BASE_DIR, "test/images/")
MASK_SAVE_DIR = "predicted_masks/"
os.makedirs(MASK_SAVE_DIR, exist_ok=True)

# --- STRATEGI PROGRESSIVE ---
PHASE_1_SIZE = 512
PHASE_1_EPOCHS = 15
PHASE_2_SIZE = 768
PHASE_2_EPOCHS = 10

# Data Preparation & Oversampling
all_files = sorted(os.listdir(TRAIN_IMG))
LIST_DICE_0 = ['train_062.jpg', 'train_065.jpg', 'train_068.jpg', 'train_075.jpg', 'train_079.jpg', 'train_084.jpg', 'train_121.jpg', 'train_134.jpg', 'train_137.jpg', 'train_160.jpg', 'train_236.jpg', 'train_263.jpg', 'train_336.jpg', 'train_352.jpg', 'train_358.jpg', 'train_363.jpg', 'train_401.jpg', 'train_424.jpg', 'train_438.jpg', 'train_455.jpg', 'train_472.jpg', 'train_474.jpg', 'train_489.jpg']
LIST_DICE_LOW = ['train_020.jpg', 'train_029.jpg', 'train_283.jpg', 'train_496.jpg']
LIST_DICE_MID = ['train_002.jpg', 'train_023.jpg', 'train_027.jpg', 'train_071.jpg', 'train_100.jpg', 'train_101.jpg', 'train_127.jpg', 'train_135.jpg', 'train_159.jpg', 'train_183.jpg', 'train_212.jpg', 'train_227.jpg', 'train_231.jpg', 'train_234.jpg', 'train_240.jpg', 'train_247.jpg', 'train_255.jpg', 'train_261.jpg', 'train_265.jpg', 'train_267.jpg', 'train_271.jpg', 'train_272.jpg', 'train_281.jpg', 'train_291.jpg', 'train_294.jpg', 'train_302.jpg', 'train_389.jpg', 'train_412.jpg', 'train_413.jpg', 'train_415.jpg', 'train_428.jpg', 'train_475.jpg', 'train_486.jpg']

very_low = LIST_DICE_0 + LIST_DICE_LOW
img_files = all_files + (very_low * 2) + (LIST_DICE_MID * 1)
random.shuffle(img_files)

mask_map = {"".join(filter(str.isdigit, m)): m for m in os.listdir(TRAIN_MASK)}

# ====================================================
# 2. ADVANCED LOSS FUNCTION (Boundary Weighted)
# ====================================================
class BoundaryWeightedLoss(torch.nn.Module):
    def __init__(self, weight_factor=3.0, kernel_size=3):
        super().__init__()
        self.weight_factor = weight_factor
        self.kernel_size = kernel_size
        self.dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=False)
        self.focal_loss = smp.losses.FocalLoss(mode='binary')

    def forward(self, pred, target):
        padding = self.kernel_size // 2
        dilated = F.max_pool2d(target, kernel_size=self.kernel_size, stride=1, padding=padding)
        eroded = -F.max_pool2d(-target, kernel_size=self.kernel_size, stride=1, padding=padding)
        boundary = dilated - eroded
        
        weight_map = torch.ones_like(target) + (boundary * (self.weight_factor - 1.0))
        
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        weighted_bce = (bce * weight_map).mean()
        
        # The Holy Trinity Combo
        return (0.3 * weighted_bce) + (0.3 * self.dice_loss(pred, target)) + (0.4 * self.focal_loss(pred, target))

# ====================================================
# 3. MODEL INITIALIZATION
# ====================================================
model = smp.Segformer(
    encoder_name="mit_b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(DEVICE)

# ====================================================
# 4. TRAINING ENGINE (Progressive Resizing)
# ====================================================
def get_train_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.CLAHE(clip_limit=2.0, p=1.0),
        ], p=0.4),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

def train_phase(model, epochs, img_size, start_lr):
    print(f"\n--- 🚀 TRAINING PHASE: {img_size}px ---")
    transform = get_train_transform(img_size)
    criterion = BoundaryWeightedLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=start_lr, weight_decay=1e-3)
    
    steps_per_epoch = (len(img_files) + BATCH_SIZE - 1) // BATCH_SIZE
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=start_lr, total_steps=steps_per_epoch * epochs,
        pct_start=0.3, div_factor=10, final_div_factor=100
    )

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(range(0, len(img_files), BATCH_SIZE), desc=f"Ep [{epoch+1}/{epochs}]")
        for i in pbar:
            batch_files = img_files[i:i+BATCH_SIZE]
            imgs_b, masks_b = [], []
            for f in batch_files:
                img = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMG, f)), cv2.COLOR_BGR2RGB)
                img_id = "".join(filter(str.isdigit, f))
                m_file = mask_map.get(img_id)
                if m_file:
                    mask = (cv2.imread(os.path.join(TRAIN_MASK, m_file), cv2.IMREAD_GRAYSCALE) > 0).astype(np.float32)
                    aug = transform(image=img, mask=mask)
                    imgs_b.append(aug["image"])
                    masks_b.append(aug["mask"].unsqueeze(0))

            if not imgs_b: continue
            imgs_t, masks_t = torch.stack(imgs_b).to(DEVICE), torch.stack(masks_b).to(DEVICE)
            
            optimizer.zero_grad()
            output = model(imgs_t)
            loss = criterion(output, masks_t)
            loss.backward()
            optimizer.step()
            scheduler.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")

# Jalankan Phase 1 (512px)
train_phase(model, PHASE_1_EPOCHS, PHASE_1_SIZE, start_lr=1e-4)

# Jalankan Phase 2 (768px Fine-tuning)
train_phase(model, PHASE_2_EPOCHS, PHASE_2_SIZE, start_lr=5e-5)

torch.save(model.state_dict(), "final_pothole_model.pth")

# ====================================================
# 5. FIND BEST THRESHOLD
# ====================================================
def find_best_threshold(model, val_files):
    model.eval()
    val_transform = A.Compose([
        A.Resize(PHASE_2_SIZE, PHASE_2_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])
    
    thresholds = np.arange(0.25, 0.6125, 0.0125)
    best_dice, best_thresh = 0, 0.5
    
    all_preds, all_gts = [], []
    print("\n🔍 Scanning for best threshold on validation samples...")
    
    with torch.no_grad():
        for f in tqdm(all_files[:60]): # Ambil 60 sampel validasi
            img = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMG, f)), cv2.COLOR_BGR2RGB)
            img_id = "".join(filter(str.isdigit, f))
            gt = (cv2.imread(os.path.join(TRAIN_MASK, mask_map[img_id]), cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)
            
            img_t = val_transform(image=img)["image"].unsqueeze(0).to(DEVICE)
            pred = model(img_t).cpu().numpy().squeeze()
            pred_res = cv2.resize(pred, (gt.shape[1], gt.shape[0]))
            all_preds.append(pred_res), all_gts.append(gt)

    for thresh in thresholds:
        dices = []
        for p, g in zip(all_preds, all_gts):
            p_bin = (p > thresh).astype(np.uint8)
            dice = (2. * (p_bin * g).sum() + 1e-7) / (p_bin.sum() + g.sum() + 1e-7)
            dices.append(dice)
        mean_d = np.mean(dices)
        if mean_d > best_dice:
            best_dice, best_thresh = mean_d, thresh
            
    print(f"✨ BEST THRESHOLD: {best_thresh:.2f} (Dice: {best_dice:.4f})")
    return best_thresh

BEST_T = find_best_threshold(model, all_files)

# ====================================================
# 6. INFERENCE (Submission)
# ====================================================
def encode_rle(mask):
    pixels = (mask == 255).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

test_transform = A.Compose([
    A.Resize(PHASE_2_SIZE, PHASE_2_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

submission = []
model.eval()
with torch.no_grad():
    for f in tqdm(sorted(os.listdir(TEST_IMG)), desc="Inference"):
        raw_img = cv2.imread(os.path.join(TEST_IMG, f))
        h, w = raw_img.shape[:2]
        img_t = test_transform(image=cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))["image"].unsqueeze(0).to(DEVICE)
        
        # TTA: Horizontal Flip
        out = model(img_t)
        out_f = torch.flip(model(torch.flip(img_t, dims=[3])), dims=[3])
        pred = ((out + out_f) / 2).cpu().numpy().squeeze()
        
        pred_full = cv2.resize(pred, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_final = (pred_full > BEST_T).astype(np.uint8) * 255
        
        submission.append({"ImageId": f, "rle": encode_rle(mask_final)})

pd.DataFrame(submission).to_csv("submission_master.csv", index=False)
print("🏁 MISSION ACCOMPLISHED: submission_master.csv saved!")


--- 🚀 TRAINING PHASE: 512px ---


Ep [15/15]: 100%|███████████████████████████████████████████| 293/293 [01:11<00:00,  4.10it/s, loss=0.1156, lr=1.0e-07]



--- 🚀 TRAINING PHASE: 768px ---


Ep [10/10]: 100%|███████████████████████████████████████████| 293/293 [02:32<00:00,  1.92it/s, loss=0.1118, lr=5.0e-08]



🔍 Scanning for best threshold on validation samples...


100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [00:07<00:00,  7.85it/s]


✨ BEST THRESHOLD: 0.41 (Dice: 0.9280)


Inference: 100%|█████████████████████████████████████████████████████████████████████| 295/295 [00:47<00:00,  6.23it/s]


🏁 MISSION ACCOMPLISHED: submission_master.csv saved!


In [3]:
def visualize_case(case, title=""):
    f = case["file"]
    dice = case["dice"]

    # 1. Load Data
    img = cv2.imread(os.path.join(TRAIN_IMG, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    img_id = "".join(filter(str.isdigit, f))
    m_file = mask_map.get(img_id)
    gt = (cv2.imread(os.path.join(TRAIN_MASK, m_file), cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)

    # 2. Prediction
    model.eval()
    with torch.no_grad():
        h, w = img.shape[:2]
        img_t = val_transform(image=img)["image"].unsqueeze(0).to(DEVICE)
        pred_raw = model(img_t).cpu().numpy().squeeze()
        pred_resized = cv2.resize(pred_raw, (w, h), interpolation=cv2.INTER_NEAREST)
        pred_mask = (pred_resized > 0.4).astype(np.uint8)

    # 3. Create Transparent Overlay
    # Kita buat layer kosong sewarna gambar asli
    overlay = img.copy()
    
    # Buat mask warna (Green untuk GT, Red untuk Pred)
    color_mask = np.zeros_like(img)
    color_mask[gt > 0] = [0, 255, 0]      # Hijau
    color_mask[pred_mask > 0] = [255, 0, 0] # Merah
    
    # Area tumpukan (Overlap) secara otomatis akan terlihat beda karena blending
    # alpha adalah opasitas gambar asli, beta adalah opasitas warna mask
    alpha = 0.6 
    beta = 0.4
    combined_overlay = cv2.addWeighted(overlay, alpha, color_mask, beta, 0)

    # --- Plotting ---
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))

    axs[0].imshow(img)
    axs[0].set_title("Original Image")

    axs[1].imshow(gt, cmap="gray")
    axs[1].set_title("Ground Truth")

    axs[2].imshow(pred_resized, cmap="gray")
    axs[2].set_title("Model Probability")

    axs[3].imshow(combined_overlay)
    axs[3].set_title("Transparent Overlay\n(Green: GT, Red: Pred)")

    plt.suptitle(f"{title} | Dice: {dice:.4f}", fontsize=16)
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [4]:
# Urutkan dulu yang terburuk
low_dice_cases_sorted = sorted([x for x in results if x["dice"] < 0.50], key=lambda x: x["dice"])

print(f"🔎 Menampilkan 5 Kasus Terburuk dari {len(low_dice_cases_sorted)} file sulit...")

for i, case in enumerate(low_dice_cases_sorted[:len(low_dice_cases_sorted)]):
    visualize_case(case, title=f"WORST #{i+1} | {case['file']}")

NameError: name 'results' is not defined

In [ ]:
# Ambil yang dice antara 0.5 sampai 0.8
mid_dice_cases_sorted = sorted([x for x in results if 0.5 <= x["dice"] < 0.8], key=lambda x: x["dice"])

print(f"🔎 Menampilkan Kasus Mid-Class (Dice 0.5 - 0.8) dari {len(mid_dice_cases_sorted)} file...")

for i, case in enumerate(mid_dice_cases_sorted[:len(mid_dice_cases_sorted)]):
    visualize_case(case, title=f"MID-CLASS #{i+1} | {case['file']}")

In [ ]:
# 1. Gabungkan list buronan very low
very_low_list = LIST_DICE_0 + LIST_DICE_LOW

# 2. Ambil skor dice dari variabel 'results' yang filenya ada di list tersebut
very_low_scores = [x["dice"] for x in results if x["file"] in very_low_list]

# 3. Hitung Mean Dice
if len(very_low_scores) > 0:
    mean_very_low = np.mean(very_low_scores)
    total_all = np.mean([x["dice"] for x in results])
    
    print("="*50)
    print(f"📊 STATISTIK KHUSUS VERY LOW CASES")
    print("="*50)
    print(f"Jumlah Gambar         : {len(very_low_scores)} file")
    print(f"Mean Dice (Very Low)  : {mean_very_low:.4f} ✨")
    print(f"Mean Dice (All Train) : {total_all:.4f}")
    print(f"Selisih Gap           : {total_all - mean_very_low:.4f}")
    
    # Cek berapa banyak yang sudah 'Lulus' (Dice > 0.5)
    passed_cases = [s for s in very_low_scores if s >= 0.5]
    print(f"Sudah Lulus (>0.5)    : {len(passed_cases)} dari {len(very_low_scores)} gambar")
    print("="*50)
else:
    print("❌ File tidak ditemukan dalam data results. Pastikan Section 5 (Evaluation) sudah dijalankan.")

In [5]:
# 1. Gabungkan list buronan very low
very_low_list = LIST_DICE_0 + LIST_DICE_LOW

# 2. Ambil skor dice dari variabel 'results' yang filenya ada di list tersebut
very_low_scores = [x["dice"] for x in results if x["file"] in very_low_list]

# 3. Hitung Mean Dice
if len(very_low_scores) > 0:
    mean_very_low = np.mean(very_low_scores)
    total_all = np.mean([x["dice"] for x in results])
    
    print("="*50)
    print(f"📊 STATISTIK KHUSUS VERY LOW CASES")
    print("="*50)
    print(f"Jumlah Gambar         : {len(very_low_scores)} file")
    print(f"Mean Dice (Very Low)  : {mean_very_low:.4f} ✨")
    print(f"Mean Dice (All Train) : {total_all:.4f}")
    print(f"Selisih Gap           : {total_all - mean_very_low:.4f}")
    
    # Cek berapa banyak yang sudah 'Lulus' (Dice > 0.5)
    passed_cases = [s for s in very_low_scores if s >= 0.5]
    print(f"Sudah Lulus (>0.5)    : {len(passed_cases)} dari {len(very_low_scores)} gambar")
    print("="*50)
else:
    print("❌ File tidak ditemukan dalam data results. Pastikan Section 5 (Evaluation) sudah dijalankan.")

NameError: name 'results' is not defined